# Differntial Fine-tuning & Smart Transfer Learning

**Theme:**

*“Not all layers should learn at the same speed.”*

Earlier we use similar learing rate for every layer but now we will use different learing rates for different layers.

- `layer4` = already good features → needs small LR

- `fc` = new random layer → needs larger LR

So we assign different LRs to different parameter groups.

## Import

In [1]:
import torch 
import torch.nn as nn
from torchvision import datasets, transforms, models 
from torch.utils.data import DataLoader 

torch.__version__

'2.8.0+cu129'

In [3]:
import torchvision
import torchaudio


print(torchvision.__version__)
torchaudio.__version__

0.23.0+cu129


'2.8.0+cu129'

## Stronger Data Augmentation



- Analyze the "Report" [here](###-Report) then decide the transform method

Why?

- Transfer learning improves when data is slightly varied.

In [ ]:
# Transform 
# transform = transforms.Compose([
#     transforms.Resize((224,224)), 
#     transforms.RandomHorizontalFlip(),
#     transforms.RandomRotation(30), 
#     transforms.ColorJitter([0.2,0.5], [0.1,0.4]),
#     transforms.ToTensor()
# ])

transform = transforms.Compose([
    transforms.Resize((224,224)), 
    transforms.ToTensor()
])

# Load datasets
train_data = datasets.CIFAR10(root='../week2/data', train=True, download=True, transform=transform) 
test_data = datasets.CIFAR10(root='../week2/data', train=False, download=True, transform=transform) 

# Dataloader 
train_loader = DataLoader(train_data, batch_size=100, shuffle=True) 
test_loader = DataLoader(test_data, batch_size=100, shuffle=False)

# Pretrained model (ResNet)
model = models.resnet18(weights=models.ResNet18_Weights.DEFAULT)

c:\Users\skroh\Desktop\Gen AI\.torchenv\Lib\site-packages\torchvision\datasets\cifar.py:83: VisibleDeprecationWarning: dtype(): align should be passed as Python or NumPy boolean but got `align=0`. Did you mean to pass a tuple to create a subarray type? (Deprecated NumPy 2.4)
  entry = pickle.load(f, encoding="latin1")


## Freezing & Unfreezing 

In [34]:
# Freezing the backbone
for param in model.parameters():
    param.requires_grad = False

# Unfreezing the layer4 
for param in model.layer4.parameters():
    param.requires_grad = True 

# defining classifier 
model.fc = nn.Linear(model.fc.in_features,10)  # fc layer is now trainable by default

## Differential learning rate (Core Concept today)

In [35]:
df_lr = [{'params':model.layer4.parameters(), 'lr':1e-5}, 
      {'params':model.fc.parameters(), 'lr':1e-3}
    ]

optimizer = torch.optim.Adam(df_lr)

| Layer  | Reason                   | LR         |
| ------ | ------------------------ | ---------- |
| layer4 | Already trained features | very small |
| fc     | Random new weights       | larger     |


## Learning rate scheduler 

We reduce the learning rate gradually

In [26]:
scheduler = torch.optim.lr_scheduler.StepLR(
    optimizer=optimizer,
    step_size=3,
    gamma=0.2
)

After epoch :
```python
scheduler.step()
```

In [27]:
# GPU availability 
if torch.cuda.is_available():
    device = 'cuda:0' 
    print("GPU is available") 
else:
    device = 'cpu' 
    print("GPU is not available") 


GPU is available


In [36]:
model = model.to(device)
loss_fn = nn.CrossEntropyLoss()

## Training Loop

### Train

You should've 20mins if you want to run the loop

In [39]:
for epoch in range(5):
    model.train()
    train_loss = 0.0
    correct = 0
    total = 0

    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)

        outputs = model(images)
        loss = loss_fn(outputs, labels)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        preds = outputs.argmax(dim=1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    #scheduler.step()

    print(f"Epoch {epoch+1} | Train Loss: {train_loss/len(train_loader):.4f} | Train Acc: {correct/total * 100:.2f}%")

Epoch 1 | Train Loss: 0.5337 | Train Acc: 81.25%
Epoch 2 | Train Loss: 0.5135 | Train Acc: 81.93%
Epoch 3 | Train Loss: 0.5017 | Train Acc: 82.53%
Epoch 4 | Train Loss: 0.4819 | Train Acc: 83.14%
Epoch 5 | Train Loss: 0.4665 | Train Acc: 83.55%


### Report

#### We use different transform for train and test data.

Result without the scheduler: 
```bash
Epoch 1 | Train Loss: 0.9062 | Train Acc: 68.19%
Epoch 2 | Train Loss: 0.7107 | Train Acc: 75.13%
Epoch 3 | Train Loss: 0.6342 | Train Acc: 77.57%
Epoch 4 | Train Loss: 0.5887 | Train Acc: 79.35%
Epoch 5 | Train Loss: 0.5577 | Train Acc: 80.36%
- Test Loss: 5.78604 | Accuracy:  11.87 %
```

Result with the shceduler:
```bash
Epoch 1 | Train Loss: 0.5319 | Train Acc: 81.47%
Epoch 2 | Train Loss: 0.5134 | Train Acc: 82.15%
Epoch 3 | Train Loss: 0.4893 | Train Acc: 82.81%
Epoch 4 | Train Loss: 0.4643 | Train Acc: 83.88%
Epoch 5 | Train Loss: 0.4571 | Train Acc: 83.90%
- Test Loss: 7.22282 | Accuracy:  11.67 %
```


#### Now we use same transform for train and test data. [Click](##-Stronger-Data-Augmentation)

##### Only resizing the images

Run without the scheduler:

```bash
Epoch 1 | Train Loss: 0.6481 | Train Acc: 78.73%
Epoch 2 | Train Loss: 0.3465 | Train Acc: 87.94%
Epoch 3 | Train Loss: 0.2632 | Train Acc: 90.94%
Epoch 4 | Train Loss: 0.2009 | Train Acc: 93.18%
Epoch 5 | Train Loss: 0.1482 | Train Acc: 95.11%
- Test Loss: 0.34428 | Accuracy:  88.82 %
```

Run with the scheduler:

```bash
Epoch 1 | Train Loss: 0.1084 | Train Acc: 96.41%
Epoch 2 | Train Loss: 0.0718 | Train Acc: 97.89%
Epoch 3 | Train Loss: 0.0459 | Train Acc: 98.89%
Epoch 4 | Train Loss: 0.0283 | Train Acc: 99.52%
Epoch 5 | Train Loss: 0.0249 | Train Acc: 99.66%
- Test Loss: 0.39353 | Accuracy:  89.40 %
```

--- 

##### Resizing, flipping, rotating 30 deg, adjusting brightness and contrast

Run without the scheduler:

```bash
Epoch 1 | Train Loss: 0.5337 | Train Acc: 81.25%
Epoch 2 | Train Loss: 0.5135 | Train Acc: 81.93%
Epoch 3 | Train Loss: 0.5017 | Train Acc: 82.53%
Epoch 4 | Train Loss: 0.4819 | Train Acc: 83.14%
Epoch 5 | Train Loss: 0.4665 | Train Acc: 83.55%
- Test Loss: 0.48994 | Accuracy:  83.47 %
```

Run with the scheduler:

```bash
Epoch 1 | Train Loss: 1.0861 | Train Acc: 62.48%
Epoch 2 | Train Loss: 0.7296 | Train Acc: 74.37%
Epoch 3 | Train Loss: 0.6448 | Train Acc: 77.33%
Epoch 4 | Train Loss: 0.5971 | Train Acc: 79.36%
Epoch 5 | Train Loss: 0.5617 | Train Acc: 80.33%
- Test Loss: 0.55126 | Accuracy:  80.90 %
```

## Evaluation

In [40]:
model.eval() 

with torch.no_grad():
    correct = 0 
    total = 0 
    test_loss = 0.0

    for images,labels in test_loader:
        images = images.to(device)
        labels = labels.to(device) 

        outputs = model(images) 
        loss = loss_fn(outputs,labels) 
        predictions = outputs.argmax(dim=1) 

        correct += (predictions==labels).sum().item() 
        total += labels.size(0) 
        test_loss += loss.item() 

    print(f"Test Loss: {test_loss/len(test_loader) :.5f} | Accuracy: {correct/total * 100 : .2f} %")

Test Loss: 0.48994 | Accuracy:  83.47 %
